# HDB Resale Flat Prices — ETL Pipeline
### Data Engineering Team | Part 1: Developing Data Pipelines

**Scope:** January 2012 – December 2016 resale flat transactions.

**Source:** [data.gov.sg — Resale Flat Prices](https://data.gov.sg/collections/189/view)

This notebook builds an end-to-end ETL pipeline that:
1. **Extracts** raw source files and unions them into a single master dataset (scope-filtered programmatically).
2. **Profiles** the data and applies **data-driven validation rules**.
3. **Cleans**: resolves composite-key duplicates, recomputes remaining lease, flags anomalous prices.
4. **Transforms**: derives the `Resale Identifier` column, resolves identifier-level duplicates.
5. **Hashes** the identifier with SHA-256 (irreversible, uniqueness-preserving).
6. **Outputs** five artefact groups: `raw`, `cleaned`, `failed`, `transformed`, `hashed`.

All reusable logic lives in `../src/*.py` modules (imported below) — this notebook is the
orchestration + documentation layer, per the assignment's emphasis on maintainable,
explainable, and robust engineering.

> **Source data note:** data.gov.sg splits this dataset across several files by date range.
> To cover Jan 2012 – Dec 2016 exactly, I need the **tail end** (Jan–Feb 2012) of the
> "2000 – Feb 2012" file, plus the full "Mar 2012 – Dec 2014" and "Jan 2015 – Dec 2016" files.
> No manually deletion of rows from any file. Hence instead, I load every raw file as is and
> apply a programmatic `month` range filter (see `extract.build_master_dataset`), which
> naturally also excludes the out-of-scope 1990–1999 file without any special-casing. Hope it clarifies the initialisation step I did
> 


## I do the setup first


In [1]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import pandas as pd
import extract, profiling, validate, dedup, lease, anomaly, transform, hashing

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RAW_DIR = "../data/raw"
OUT_DIR = "../outputs"


## 1. Extraction
Load every raw CSV file exactly as downloaded (no manual edits), copy them into
`outputs/raw/`, then union all files' columns and filter programmatically
to the Jan 2012 – Dec 2016 scope. 

In [19]:
raw_frames = extract.load_raw_files(RAW_DIR)
for name, df in raw_frames.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
# Exploratory data analysis 

Resale_Flat_Prices__Based_on_Approval_Date___1990_-_1999.csv: 287,196 rows, 10 columns
Resale_Flat_Prices__Based_on_Approval_Date___2000_-_Feb_2012.csv: 369,651 rows, 10 columns
Resale_Flat_Prices__Based_on_Registration_Date___From_Jan_2015_to_Dec_2016.csv: 37,153 rows, 11 columns
Resale_Flat_Prices__Based_on_Registration_Date___From_Mar_2012_to_Dec_2014.csv: 52,203 rows, 10 columns


In [20]:
# Persist raw files as-is into outputs/raw/ (audit trail — untouched copies)
# Copy all downloaded raw files into raw data directory
import shutil
for name in raw_frames:
    shutil.copy(os.path.join(RAW_DIR, name), os.path.join(OUT_DIR, "raw", name))
print("Raw files copied to outputs/raw/")


Raw files copied to outputs/raw/


In [4]:
master = extract.build_master_dataset(raw_frames)
print("Master dataset shape (scope-filtered, Jan 2012 - Dec 2016):", master.shape)
print("Month range:", master['month'].min(), "to", master['month'].max())
print("\nColumns (union of all source files):", master.columns.tolist())
master.head()


Master dataset shape (scope-filtered, Jan 2012 - Dec 2016): (92544, 11)
Month range: 2012-01 to 2016-12

Columns (union of all source files): ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease']


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
0,2012-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,257800.0,NaN
1,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,263000.0,NaN
2,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,275000.0,NaN
3,2012-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,260000.0,NaN
4,2012-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,226000.0,NaN


## 2. Data Profiling
A lightweight profiler (no external dependency) summarising, per column: dtype, null
count/percentage, distinct value count, and either numeric summary stats or top categorical
values. This report grounds the validation rules below in the dataset's actual statistical
properties, as required by the assignment.

In [5]:
profile_report = profiling.profile_dataframe(master)
profile_report


,column,dtype,n_null,pct_null,n_distinct,min,mean,median,max,std,top_values
0,month,str,0,0.00,60,NaN,NaN,NaN,NaN,NaN,2012-03 (2360); 2012-05 (2323); 2012-07 (2179)...
1,town,str,0,0.00,26,NaN,NaN,NaN,NaN,NaN,JURONG WEST (7573); WOODLANDS (7399); TAMPINES...
2,flat_type,str,0,0.00,7,NaN,NaN,NaN,NaN,NaN,4 ROOM (36535); 3 ROOM (26307); 5 ROOM (21368)...
3,block,str,0,0.00,2139,NaN,NaN,NaN,NaN,NaN,2 (397); 1 (350); 108 (318); 107 (309); 113 (298)
4,street_name,str,0,0.00,522,NaN,NaN,NaN,NaN,NaN,YISHUN RING RD (1629); BEDOK RESERVOIR RD (126...
5,storey_range,str,0,0.00,25,NaN,NaN,NaN,NaN,NaN,04 TO 06 (21214); 07 TO 09 (18765); 01 TO 03 (...
6,floor_area_sqm,float64,0,0.00,168,31.0,96.57,95.0,280.0,24.68,NaN
7,flat_model,str,0,0.00,20,NaN,NaN,NaN,NaN,NaN,Model A (26447); Improved (24117); New Generat...
8,lease_commence_date,int64,0,0.00,48,1966.0,1990.07,1988.0,2013.0,10.45,NaN
9,resale_price,float64,0,0.00,2615,190000.0,450938.97,428000.0,1150000.0,128181.30,NaN


In [6]:
profile_report.to_csv(os.path.join(OUT_DIR, "profiling_report.csv"), index=False)
print("Saved profiling report.")


Saved profiling report.


## 3. Data Validation
Validation rules for `month`, `town`, `flat_type`, `flat_model`, and `storey_range` are
derived from the master dataset's own observed values / formats (see `src/validate.py`
docstring for the exact rule for each field), rather than an externally hardcoded list.
We also add two extra sanity rules (`floor_area_sqm` and `resale_price` must be positive
and plausible) as additional data quality checks per the assignment's open-ended requirement.

Rows failing **any** rule are routed to the `failed` output with a reason tag; rows passing
all rules proceed to cleaning.

In [7]:
validated = validate.run_all_validations(master)

rule_cols = ['valid_month','valid_town','valid_flat_type','valid_flat_model',
             'valid_storey_range','valid_floor_area','valid_resale_price']
print("Validation pass rate:", round(validated['is_valid'].mean() * 100, 2), "%")
for c in rule_cols:
    n_fail = (~validated[c]).sum()
    print(f"  {c}: {n_fail} failing rows")


Validation pass rate: 100.0 %
  valid_month: 0 failing rows
  valid_town: 0 failing rows
  valid_flat_type: 0 failing rows
  valid_flat_model: 0 failing rows
  valid_storey_range: 0 failing rows
  valid_floor_area: 0 failing rows
  valid_resale_price: 0 failing rows


In [8]:
valid_rows = validated[validated['is_valid']].drop(columns=rule_cols + ['is_valid']).reset_index(drop=True)
invalid_rows = validated[~validated['is_valid']].copy()

# Tag WHY each invalid row failed (which rule(s))
def failure_reasons(row):
    return "; ".join(c.replace("valid_", "") + "_invalid" for c in rule_cols if not row[c])

if len(invalid_rows):
    invalid_rows['failure_reason'] = invalid_rows.apply(failure_reasons, axis=1)
    invalid_rows = invalid_rows.drop(columns=rule_cols + ['is_valid'])

print("Valid rows proceeding to cleaning:", len(valid_rows))
print("Invalid rows routed to failed/:", len(invalid_rows))


Valid rows proceeding to cleaning: 92544
Invalid rows routed to failed/: 0


## 4. Cleaning

### 4.1 Composite-key duplicate resolution
Per the assignment, the composite key is *all columns except resale_price*. I additionally
exclude the source-provided `remaining_lease` column from the key, since I discard and
recompute that field in step 4.2 (it is not part of a row's business identity —
see `src/dedup.py` docstring). Where duplicate keys exist, I keep the higher-priced row and
route the rest to `failed`.

In [9]:
cleaned, dup_failed = dedup.resolve_duplicates(valid_rows)
print("Rows after composite-key dedup (cleaned):", len(cleaned))
print("Rows discarded to failed/ (lower-price duplicates):", len(dup_failed))


Rows after composite-key dedup (cleaned): 90944
Rows discarded to failed/ (lower-price duplicates): 1600


### 4.2 Remaining lease recomputation
Assume a 99-year HDB lease. Since `lease_commence_date` only provides a year, we assume the
lease commences 1 January of that year, and compute remaining lease **as of today**, floored
down to whole years + months (see `src/lease.py` docstring for the exact assumption).

In [10]:
cleaned = lease.add_remaining_lease_columns(cleaned, lease_col="lease_commence_date")
cleaned[['lease_commence_date','remaining_lease_years','remaining_lease_months','remaining_lease_display']].head()


,lease_commence_date,remaining_lease_years,remaining_lease_months,remaining_lease_display
0,1979,51,5,51 years 5 months
1,1978,50,5,50 years 5 months
2,1978,50,5,50 years 5 months
3,1986,58,5,58 years 5 months
4,1986,58,5,58 years 5 months


### 4.3 Anomalous resale price detection
**Heuristic:** compute a Z-score for each row's `resale_price` within its peer group
(`town` + `flat_type` + `flat_model`), since price ranges differ enormously across town/type/
model combinations. Rows with `|Z| > 3` (and peer group size ≥ 5, so the standard deviation
is statistically meaningful) are flagged `is_price_anomaly = True`.

**Assumption:** anomalies are flagged for downstream analyst awareness (an extra column), not
silently dropped — the assignment doesn't ask us to remove them, only to identify them.

In [11]:
cleaned = anomaly.flag_price_anomalies(cleaned)
print("Anomalous rows flagged:", cleaned['is_price_anomaly'].sum(), "/", len(cleaned))
cleaned[cleaned['is_price_anomaly']][['month','town','flat_type','flat_model','resale_price','price_zscore']].head(10)


Anomalous rows flagged: 267 / 90944


,month,town,flat_type,flat_model,resale_price,price_zscore
261,2012-01,BUKIT MERAH,2 ROOM,Standard,442000.0,3.675682
890,2012-01,PASIR RIS,5 ROOM,Model A,648000.0,3.386565
904,2012-01,PASIR RIS,EXECUTIVE,Maisonette,840000.0,3.008414
3591,2015-01,GEYLANG,3 ROOM,Model A,340000.0,-3.032224
4184,2015-01,TAMPINES,5 ROOM,DBSS,550000.0,-3.475660
4776,2015-02,GEYLANG,4 ROOM,Simplified,488000.0,3.846311
4987,2015-02,KALLANG/WHAMPOA,3 ROOM,New Generation,532000.0,3.076583
6906,2015-04,ANG MO KIO,3 ROOM,New Generation,465000.0,3.110620
7631,2015-04,JURONG WEST,4 ROOM,Model A,550000.0,3.663239
7748,2015-04,KALLANG/WHAMPOA,5 ROOM,Standard,800000.0,3.182743


### 4.4 Save cleaned & failed outputs

In [12]:
all_failed = pd.concat([invalid_rows, dup_failed], ignore_index=True, sort=False)

cleaned.to_csv(os.path.join(OUT_DIR, "cleaned", "cleaned_resale_flat_prices.csv"), index=False)
all_failed.to_csv(os.path.join(OUT_DIR, "failed", "failed_records.csv"), index=False)

print("Cleaned dataset saved:", cleaned.shape)
print("Failed dataset saved:", all_failed.shape)
all_failed['failure_reason'].value_counts()


Cleaned dataset saved: (90944, 16)
Failed dataset saved: (1600, 20)


failure_reason
duplicate_composite_key_lower_price    1600
Name: count, dtype: int64

## 5. Transformation

### 5.1 Resale Identifier
Derived per the assignment's exact 9-character spec — see `src/transform.py` docstring for
the full breakdown of each character group (block digits, peer-group average price digits,
row's own month, town initial).

In [13]:
transformed = transform.add_resale_identifier(cleaned)
transformed[['month','town','block','flat_type','resale_price','resale_identifier']].head(10)


,month,town,block,flat_type,resale_price,resale_identifier
0,2012-01,ANG MO KIO,406,2 ROOM,257800.0,S4062501A
1,2012-01,ANG MO KIO,314,2 ROOM,263000.0,S3142501A
2,2012-01,ANG MO KIO,314,2 ROOM,275000.0,S3142501A
3,2012-01,ANG MO KIO,170,2 ROOM,260000.0,S1702501A
4,2012-01,ANG MO KIO,174,2 ROOM,226000.0,S1742501A
5,2012-01,ANG MO KIO,508,2 ROOM,260000.0,S5082501A
6,2012-01,ANG MO KIO,174,3 ROOM,281000.0,S1743401A
7,2012-01,ANG MO KIO,216,3 ROOM,375000.0,S2163401A
8,2012-01,ANG MO KIO,332,3 ROOM,350000.0,S3323401A
9,2012-01,ANG MO KIO,418,3 ROOM,420000.0,S4183401A


In [14]:
# sanity check: identifier is always exactly 9 characters, starts with 'S'
assert (transformed['resale_identifier'].str.len() == 9).all()
assert (transformed['resale_identifier'].str[0] == 'S').all()
print("Resale Identifier format check passed.")


Resale Identifier format check passed.


### 5.2 Resolve identifier-level duplicates
Per the assignment's step 2: if the derived identifier is not unique across rows, keep the
higher-priced row and discard the rest to `failed`.

In [15]:
transformed, id_dup_failed = transform.resolve_identifier_duplicates(transformed)
print("Rows after identifier dedup (transformed):", len(transformed))
print("Rows discarded to failed/ (identifier duplicates):", len(id_dup_failed))

# append these to the failed dataset too
all_failed = pd.concat([all_failed, id_dup_failed], ignore_index=True, sort=False)
all_failed.to_csv(os.path.join(OUT_DIR, "failed", "failed_records.csv"), index=False)

transformed.to_csv(os.path.join(OUT_DIR, "transformed", "transformed_resale_flat_prices.csv"), index=False)
print("Transformed dataset saved:", transformed.shape)


Rows after identifier dedup (transformed): 77255
Rows discarded to failed/ (identifier duplicates): 13689
Transformed dataset saved: (77255, 17)


## 6. Hashing
Hash `resale_identifier` using **SHA-256** — irreversible, and with a 256-bit output space,
collisions across our dataset size are astronomically unlikely, so uniqueness is preserved.
See `src/hashing.py` docstring for the full rationale and its limitations.

In [16]:
hashed = hashing.add_hashed_identifier(transformed)
print("Unique identifiers:", hashed['resale_identifier'].nunique())
print("Unique hashes:      ", hashed['hashed_identifier'].nunique())
print("Rows:                ", len(hashed))

hashed.to_csv(os.path.join(OUT_DIR, "hashed", "hashed_resale_flat_prices.csv"), index=False)
hashed[['resale_identifier','hashed_identifier']].head()


Unique identifiers: 77255
Unique hashes:       77255
Rows:                 77255


,resale_identifier,hashed_identifier
0,S4062501A,e22d10c12612bb3c3d1ff735a675db47039477c6ce8dbe...
1,S3142501A,df36be696d8b9d99bff955089fa0d1ee5cc92ae167f78a...
2,S1702501A,7b76c2e5ad415d73d9ac4e74b364f51dc0b60f0db59b52...
3,S1742501A,0509ae6467b1e75c84fe77c74c7f8857f8926009d7b912...
4,S5082501A,2fc7d3dd15e6bbb2178269cc278c2a995c10378f770d5d...


## 7. Summary

In [17]:
summary = pd.DataFrame({
    "stage": ["raw (union, scope-filtered)", "valid (passed validation)", "invalid (failed validation)",
              "cleaned (post composite-key dedup)", "failed (composite-key duplicates)",
              "transformed (post identifier dedup)", "failed (identifier duplicates)",
              "hashed (final output)"],
    "row_count": [len(master), len(valid_rows), len(invalid_rows),
                  len(cleaned), len(dup_failed),
                  len(transformed), len(id_dup_failed),
                  len(hashed)]
})
summary


,stage,row_count
0,"raw (union, scope-filtered)",92544
1,valid (passed validation),92544
2,invalid (failed validation),0
3,cleaned (post composite-key dedup),90944
4,failed (composite-key duplicates),1600
5,transformed (post identifier dedup),77255
6,failed (identifier duplicates),13689
7,hashed (final output),77255


In [18]:
summary.to_csv(os.path.join(OUT_DIR, "pipeline_summary.csv"), index=False)
print("Pipeline complete. All outputs saved under outputs/{raw,cleaned,failed,transformed,hashed}/")


Pipeline complete. All outputs saved under outputs/{raw,cleaned,failed,transformed,hashed}/
